# Kafka Producer B: Random Number Stream

This notebook creates a second Kafka producer for teaching stream joins.

It sends random numbers to the Kafka topic `teaching-stream-B`. The message format is intentionally the same as Producer A, except `stream_id` is `B`.

Run this notebook alongside Producer A. Leave the final cell running while the Spark Structured Streaming notebook reads and joins both streams.

In [ ]:
import json
import random
import time
import uuid
from datetime import datetime

from kafka3 import KafkaProducer

## Configuration

Use the same Kafka broker address as Producer A.

In [ ]:
HOST_IP = "10.192.8.146"
KAFKA_TOPIC = "teaching-stream-B"

producer = KafkaProducer(
    bootstrap_servers=[f"{HOST_IP}:9092"],
    value_serializer=lambda value: json.dumps(value).encode("utf-8"),
    api_version=(0, 10)
)

print(f"Producer B connected to Kafka topic: {KAFKA_TOPIC}")

## Stream Random Numbers

This producer also sends numbers from 1 to 5. Because both streams use the same small range, the Spark join notebook should regularly find matching numbers within the join window.

Stop this cell manually when you are finished.

In [ ]:
try:
    while True:
        event = {
            "event_id": str(uuid.uuid4()),
            "stream_id": "B",
            "number": random.randint(1, 5),
            "event_time": datetime.utcnow().isoformat()
        }

        producer.send(KAFKA_TOPIC, value=event)
        producer.flush()

        print(f"Sent from B: {event}")
        time.sleep(2)

except KeyboardInterrupt:
    print("Producer B stopped by user.")

finally:
    producer.close()
    print("Producer B connection closed.")